In [1]:
from primaite.agents.aegis.gllm import GLLM
from primaite.agents.aegis.modules.openai import OpenAIClient
from primaite.agents.git_agent import GITAgent
from torch.utils.data import DataLoader
from primaite.agents.llm.utils import network_connectivity_desc
import logging

logging.disable(logging.CRITICAL)

/home/jonathan/projects/primaite/PrimAITE/aegis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-08-21 16:01:11.233414: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-21 16:01:11.281062: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-21 16:01:11.974374: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/jonathan/projects/primaite/PrimAITE/aegis/lib/python3.10/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_id" has 

In [2]:
gllm = GLLM()
openai = OpenAIClient(openai_api_key="sk-")

In [3]:
questions = [
    "How many nodes are in the network?",
    "How many unique node types are there?",
]

In [4]:
# Mock a primaite graph for development
agent = GITAgent(
    training_config_path="../src/primaite/config/_package_data/training/git.yaml",
    lay_down_config_path="../src/primaite/config/_package_data/lay_down/lay_down_config_6_data_manipulation.yaml",
)
obs = agent._env.reset()
data = agent.create_graph(obs)
network_desc = network_connectivity_desc(agent._env)

<Figure size 640x480 with 0 Axes>

In [5]:
import os
import pickle as pkl

if "openai_responses.pkl" not in os.listdir("./"):
    openai_responses = []
    openai_prompts = gllm.build_prompts(questions=questions, network_desc=network_desc, model="openai")
    for prompt in openai_prompts:
        openai_responses.append(openai.generate(prompt=prompt))

    with open("openai_responses.pkl", "wb") as file:
        pkl.dump(openai_responses, file)
else:
    with open("openai_responses.pkl", "rb") as file:
        openai_responses = pkl.load(file)

In [6]:
from primaite.agents.aegis.data import GLLMDataset, collate_fn

dataset = GLLMDataset(
    graphs=[data for _ in range(len(questions))],
    questions=[question for question in questions],
    gt_answers=[response for response in openai_responses],
    llm=gllm.llm,
)

In [7]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)

In [17]:
for batch in dataloader:
    print(batch)

{'graphs': DataBatch(edge_index=[2, 36], id=[2], num_nodes=18, x=[18, 6], batch=[18], ptr=[3]), 'questions': ['How many nodes are in the network?', 'How many unique node types are there?'], 'gt_answers': ['There are 9 nodes in the network.', '4'], 'gt_hidden_states': [tensor([[ 0.1203, -0.0072,  0.2637,  ..., -0.0105, -0.0940, -0.1460]]), tensor([[ 0.0623, -0.0857,  0.1428,  ...,  0.0078,  0.0095, -0.0941]])]}


In [8]:
inputs_embeds = gllm.llm.get_input_embeddings(
    prompts=[
        "hey there, who are you?",
        "hi how are you doing? Answer:",
        "Say as many swear words in italian as you can. GO: ",
    ]
)

In [9]:
inputs_embeds.shape

torch.Size([3, 15, 2048])

In [15]:
next_token_ids, next_token_probs = gllm.llm.generate_from_embeddings(
    text_embeddings=inputs_embeds, restrict_output=True, max_new_tokens=5
)

last tokens: [40, 40, 33]
last tokens: [32, 32, 32]
last tokens: [32, 32, 32]
last tokens: [32, 32, 32]


In [16]:
gllm.llm.tokenizer.batch_decode(list(next_token_ids))

['80000', '80000', '10000']

In [ ]:
output = gllm.llm.model.forward(inputs_embeds=inputs_embeds)

In [ ]:
output[1]["hidden_states"][-1][:, -1, :].shape

In [ ]:
inputs_embeds.shape

In [ ]:
output[0].shape

In [ ]:
logits = output[1]["logits"][:, -1]
logits.shape

In [ ]:
token_ids, probs = gllm.llm._filter_logits(logits=logits, prev_token_ids=[30, 30])

In [ ]:
token_ids.shape

In [ ]:
probs.shape

In [ ]:
new_embeds = gllm.llm.get_input_embeddings(token_ids=token_ids)
new_embeds.shape

In [ ]:
inputs_embeds.shape

In [ ]:
torch.cat([inputs_embeds, new_embeds], dim=1).shape

In [ ]:
def process_logits(logits, filtered_vocab):
    # Convert filtered_vocab to a set for O(1) lookup time
    filtered_vocab_set = set(filtered_vocab.values())

    # Get the indices of tokens allowed by filtered_vocab
    allowed_indices = torch.tensor([idx for idx in range(logits.shape[-1]) if idx in filtered_vocab_set])

    # Filter logits: keep only those allowed by filtered_vocab
    filtered_logits = logits[:, allowed_indices]

    # Get the highest probability token for each sequence
    max_probs, max_indices = torch.max(filtered_logits, dim=1)

    # Map max_indices back to the original vocab indices
    token_ids = allowed_indices[max_indices.to("cpu")].unsqueeze(1)

    return token_ids, max_probs.unsqueeze(1)

In [ ]:
token_ids, probs = process_logits(logits, filtered_vocab=gllm.llm.filtered_vocab)

In [ ]:
token_ids.shape

In [ ]:
probs.shape

In [ ]:
gllm.llm.tokenizer.decode(34)

In [ ]:
you are working on the GLLM forward pass. You got as far as the generate from embeddings function in LLM.py. Note that you are generating now for N sets of embeddings rather than with one set, so it's probably for the best that you' go through the generate steps in the notebook with a set of embeddings and make adjustments as required.

In [ ]:
from primaite.agents.aegis.gllm import train_loop

gllm_responses, openai_responses = train_loop(
    model=gllm,
    dataloader=dataloader,
    network_desc=network_desc,
)